The Sequence to Sequence (Seq2Seq) model is a type of neural network architecture widely used in machine learning for tasks that involve mapping one sequence of data to another. It processes an input sequence and generates a corresponding output sequence. Seq2Seq models have had significant impact in areas such as natural language processing (NLP)  , machine translation , speech recognition and time-series prediction


Both input and output are treated as sequence of varying lengths and the model is composed of two parts

###1 Encoder
* Process the input sequence token by token.
* Encodes the entire sequence into a fixed-length  contect vector (or a series of hidden states) that summarizes the important information from the input

### Decoder

* Takes the context vector as input
* Generates the output sequence one token at a time. Predicting each token based on the context vector and previously generated tokens

The model is commonly used in tasks where there is a need to map sequences of varying lengths such as converting a sentence in one language to another or predicting a sequence of future events based on past data i.e time-series forecasting.


##Seq2Seq with RNNs

In the simplest Seq2Seq model RNNs are used in both the encoder and decoder to process sequential data. For a given input sequence (x1,x2,...,xT)(x1​,x2​,...,xT​), a RNN generates a sequence of outputs (y1,y2,...,yT)(y1​,y2​,...,yT​)​


###Limitations of Vanilla RNNs:
* Vanilla RNNs struggle with long-term dependencies due to the vanishing gradient
problem.
* To overcome this, Advanced RNN variants like LSTM or GRU (Gated Recurrent UNit ) are used in Seq2Seq models.
These architectures are better at capturing long-range dependencies


### Teacher forcing:
During training teacher forcing is commonly used. Instead  of feeding the decoder's own previous prediction as the next input, the actual target token from the training data is provided
This **Benefits**:
> Accelerating training
> Reduce error propagation



###step by step seq2seq implementation


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

### Step2: Encoder

In [4]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, hidden = self.rnn(embedded)
        return hidden

###Step 3: Decoder

In [5]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, input, hidden):
        input = input.unsqueeze(0)
        embedded = self.embedding(input)
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.fc(output.squeeze(0))
        return prediction, hidden

###Step 4: Seq2Seq Model with Teacher Forcing

    * Batch size & vocab size: extracted from input and decoder.
    * Encoding: input sequence → encoder → context vector (hidden).
    * Start token: initialize decoder with token 0.
    * Loop over max_len:
    * Decoder predicts next token.
    top1 → token with max probability.
    * Append top1 to outputs.
    * Teacher forcing: sometimes feed true target token instead of prediction.
    * Return predictions: concatenated sequence of token IDs.



In [10]:
class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder, device):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder
    self.device = device

  def forward(self, src, trg=None, max_len=10, teacher_forcing_ratio=0.5):
    batch_size = src.shape[1]
    trg_vocab_size = self.decoder.fc.out_features
    outputs = []

    hidden = self.encoder(src)

    input = torch.zeros(batch_size, dtype=torch.long).to(self.device)

    for t in range(max_len):
      output , hidden = self.decoder(input, hidden)
      top1 = output.argmax(1)
      outputs.append(top1.unsqueeze(0))

      if trg is not None and t < trg.shape[0] and torch.rand(1).item() < teacher_forcing_ratio:
        input = trg[1]
      else:
        input = top1

      outputs = torch.cat(outputs, dim=0)
      return outputs

###usage example with Outputs


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCAB_SIZE = 10
EMB_DIM = 8
HID_DIM = 16
SEQ_LEN = 5
BATCH_SIZE = 2

enc =  Encoder(VOCAB_SIZE, EMB_DIM, HID_DIM)
dec = Decoder(VOCAB_SIZE, EMB_DIM, HID_DIM)
model = Seq2Seq(enc, dec, device).to(device)

src = torch.randint(1, VOCAB_SIZE, (SEQ_LEN , BATCH_SIZE)).to(device)
trg = torch.randint(1, VOCAB_SIZE, (SEQ_LEN, BATCH_SIZE)).to(device)

outputs = model(src, trg, max_len=SEQ_LEN, teacher_forcing_ratio=0.7)

print("Source sequence (input tokens):")
print(src.T)
print("\nTarget sequence (true tokens):")
print(trg.T)
print("\nPredicted sequence (model output toekns):")
print(outputs.T)

Source sequence (input tokens):
tensor([[1, 1, 5, 3, 6],
        [5, 4, 2, 6, 2]])

Target sequence (true tokens):
tensor([[2, 2, 2, 7, 2],
        [7, 8, 3, 1, 5]])

Predicted sequence (model output toekns):
tensor([[3],
        [3]])
